# FPGA-friendly LogisticRegression: top4 rank55

This notebook recreates the fixed 55-feature LogisticRegression classifier from `fft3_basic_pattern_features_blind_v1.pkl`.

Design constraints:
- the original blind feature set contains 445 values;
- the FPGA-friendly candidate set keeps only top1-top4 peak features;
- individual grid samples (`*_grid_k...`) and symmetry scores are excluded;
- compact grid summaries, nonzero counts and peak gaps are retained;
- the final fixed 55-feature order is identical to `vectors["lr55"]` used by `Classfier-5basic-pattern-LogisticRegression-fusion-7-55-445-CM-only-ptvectors.ipynb`;
- train/test splits and LogisticRegression settings are unchanged.


In [1]:
from pathlib import Path
import sys


def _find_helper_project_dir(start=Path.cwd()):
    start = Path(start).resolve()
    search_roots = [start, *start.parents]
    for root in search_roots:
        if (root / "helper_function" / "__init__.py").exists():
            return root
    for root in search_roots[:3]:
        matches = list(root.glob("**/boi_shift_fft3_cfpattern_spl/helper_function/__init__.py"))
        if matches:
            return matches[0].parent.parent
    raise FileNotFoundError("Could not find helper_function from the current Jupyter working directory.")


PROJECT_DIR = _find_helper_project_dir()
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from pathlib import Path
import re
import time
import warnings

import numpy as np
import pandas as pd

from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix

warnings.filterwarnings("ignore")

BASE = PROJECT_DIR
CACHE_PATH = BASE / "fft3_basic_pattern_features_blind_v1.pkl"
LABEL_ORDER = ["BPSK", "MSK", "DQPSK", "8PSK", "QPSK-like"]

print("CACHE_PATH:", CACHE_PATH)
print("exists:", CACHE_PATH.exists())


CACHE_PATH: /home/ruilin/SPL_Surreal/Surreal_source/chad-reproduce/chad-reproduce-sensors/BOI+SSCA+CC-Blind-generateselectedBOI+SSCA-metadata/BOI+FFT-2018+2022-generate/boi_shift_fft3_cfpattern_finalresult/fft3_basic_pattern_features_blind_v1.pkl
exists: True


In [2]:
df = pd.read_pickle(CACHE_PATH).reset_index(drop=True)
y = df["label5"].astype(str)

META_COLS = {
    "year", "pt_name", "batch_id", "signal_index",
    "signal_type", "label5", "snr",
}
forbidden_cols = [c for c in df.columns if "truth" in c.lower() or "debug" in c.lower()]
blind445_features = [c for c in df.columns if c not in META_COLS and c not in forbidden_cols]

assert len(blind445_features) == 445, len(blind445_features)
print("data shape:", df.shape)
print("blind feature count:", len(blind445_features))
print(df.groupby(["year", "label5"]).size())


data shape: (223632, 457)
blind feature count: 445
year  label5   
2018  8PSK         13954
      BPSK         13954
      DQPSK        13954
      MSK          13954
      QPSK-like    55816
2022  8PSK         14000
      BPSK         14000
      DQPSK        14000
      MSK          14000
      QPSK-like    56000
dtype: int64


## FPGA-friendly top4 candidate set

The rule below reproduces 133 candidates: retain `rs_est`, top1-top4 features, nonzero summaries, peak gaps, center frequency and compact grid summaries; remove top5-top20 features, individual grid samples and the six symmetry scores.


In [3]:
def build_fpga_top4_candidates(feature_names):
    candidates = []
    for name in feature_names:
        top_match = re.search(r"_top(\d+)_", name)
        if top_match and int(top_match.group(1)) > 4:
            continue
        if re.search(r"_grid_k[+-]\d+_db$", name):
            continue
        if name.endswith("_sym_score_mean") or name.endswith("_sym_score_min"):
            continue
        candidates.append(name)
    return candidates

fpga_top4_candidates = build_fpga_top4_candidates(blind445_features)
assert len(fpga_top4_candidates) == 133, len(fpga_top4_candidates)
print("FPGA-friendly candidate count:", len(fpga_top4_candidates))


FPGA-friendly candidate count: 133


## Fixed rank55 selection

This is the deployed 55-feature order stored as `vector_feature_names["lr55"]` in the compact PT files.


In [4]:
rank55_features = [
    "x4_top1_minus_top2_db",
    "x4_top2_db",
    "x4_top3_db",
    "x4_top1_af",
    "x4_top1_db",
    "x4_int_grid_3rd_db",
    "x4_top1_minus_top5_db",
    "x4_top4_db",
    "x2_nonzero_count_gt_m10",
    "x2_nonzero_count_gt_m20",
    "x4_int_grid_2nd_db",
    "x2_top4_db",
    "x2_top1_minus_top5_db",
    "x2_top3_db",
    "x2_half_grid_best_db",
    "x2_half_grid_2nd_db",
    "x4_nonzero_count_gt_m10",
    "x2_top1_db",
    "abs_x2_top4_db",
    "abs_x2_top1_minus_top5_db",
    "x4_int_grid_count_gt_m10",
    "x2_top1_minus_top2_db",
    "x2_top2_db",
    "x4_top2_af",
    "x4_half_grid_count_gt_m10",
    "x2_half_grid_count_gt_m20",
    "x2_half_grid_count_gt_m30",
    "x2_top1_af",
    "abs_x2_nonzero_count_gt_m30",
    "x4_top1_af_over_rs",
    "x2_top3_af",
    "x2_int_grid_count_gt_m20",
    "x2_int_grid_2nd_db",
    "x2_top2_af",
    "abs_x2_nonzero_count_gt_m20",
    "abs_x2_top1_minus_top2_db",
    "abs_x2_nonzero_best_db",
    "abs_x2_top2_db",
    "x2_int_grid_count_gt_m30",
    "x2_half_grid_count_gt_m10",
    "x4_top3_af",
    "abs_x2_top3_db",
    "x4_half_grid_3rd_db",
    "x2_nonzero_count_gt_m30",
    "x4_nonzero_count_gt_m20",
    "x2_nonzero_best_db",
    "abs_x2_top4_af",
    "x4_half_grid_count_gt_m20",
    "x2_half_grid_3rd_db",
    "x4_top4_af",
    "x4_half_minus_int_top3_db",
    "x4_half_grid_best_db",
    "abs_x2_half_grid_mean_db",
    "abs_x2_int_grid_mean_db",
    "x4_half_grid_mean_top3_db"
]

assert len(rank55_features) == 55, len(rank55_features)
assert len(set(rank55_features)) == 55
missing_from_pkl = [c for c in rank55_features if c not in df.columns]
not_fpga_candidate = [c for c in rank55_features if c not in fpga_top4_candidates]
assert not missing_from_pkl, missing_from_pkl
assert not not_fpga_candidate, not_fpga_candidate

print("rank55 feature count:", len(rank55_features))
for i, name in enumerate(rank55_features, start=1):
    print(f"{i:02d}. {name}")


rank55 feature count: 55
01. x4_top1_minus_top2_db
02. x4_top2_db
03. x4_top3_db
04. x4_top1_af
05. x4_top1_db
06. x4_int_grid_3rd_db
07. x4_top1_minus_top5_db
08. x4_top4_db
09. x2_nonzero_count_gt_m10
10. x2_nonzero_count_gt_m20
11. x4_int_grid_2nd_db
12. x2_top4_db
13. x2_top1_minus_top5_db
14. x2_top3_db
15. x2_half_grid_best_db
16. x2_half_grid_2nd_db
17. x4_nonzero_count_gt_m10
18. x2_top1_db
19. abs_x2_top4_db
20. abs_x2_top1_minus_top5_db
21. x4_int_grid_count_gt_m10
22. x2_top1_minus_top2_db
23. x2_top2_db
24. x4_top2_af
25. x4_half_grid_count_gt_m10
26. x2_half_grid_count_gt_m20
27. x2_half_grid_count_gt_m30
28. x2_top1_af
29. abs_x2_nonzero_count_gt_m30
30. x4_top1_af_over_rs
31. x2_top3_af
32. x2_int_grid_count_gt_m20
33. x2_int_grid_2nd_db
34. x2_top2_af
35. abs_x2_nonzero_count_gt_m20
36. abs_x2_top1_minus_top2_db
37. abs_x2_nonzero_best_db
38. abs_x2_top2_db
39. x2_int_grid_count_gt_m30
40. x2_half_grid_count_gt_m10
41. x4_top3_af
42. abs_x2_top3_db
43. x4_half_grid_3rd_

In [5]:
def make_split(split_name):
    year = df["year"].astype(str)
    if split_name == "2018train20":
        train_mask = (year == "2018") & (df["batch_id"] <= 20)
        test_mask = ((year == "2018") & (df["batch_id"] > 20)) | (year == "2022")
    elif split_name == "2022train20":
        train_mask = (year == "2022") & (df["batch_id"] <= 20)
        test_mask = ((year == "2022") & (df["batch_id"] > 20)) | (year == "2018")
    else:
        raise ValueError(split_name)
    return train_mask, test_mask


def make_logreg_model():
    return make_pipeline(
        SimpleImputer(strategy="median"),
        StandardScaler(),
        LogisticRegression(
            max_iter=1000,
            n_jobs=-1,
            class_weight="balanced",
            C=3,
            random_state=17,
        ),
    )


In [6]:
X = df[rank55_features].replace([np.inf, -np.inf], np.nan).astype("float32")
result_rows = []
confusion_matrices = {}
models = {}

for split_name in ["2018train20", "2022train20"]:
    train_mask, test_mask = make_split(split_name)
    X_train, X_test = X.loc[train_mask], X.loc[test_mask]
    y_train, y_test = y.loc[train_mask], y.loc[test_mask]

    model = make_logreg_model()
    t0 = time.time()
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    elapsed = time.time() - t0

    acc = accuracy_score(y_test, pred)
    errors = int((pred != y_test.to_numpy()).sum())
    cm = pd.DataFrame(
        confusion_matrix(y_test, pred, labels=LABEL_ORDER),
        index=LABEL_ORDER,
        columns=LABEL_ORDER,
    )

    result_rows.append({
        "feature_set": "top4_rank55",
        "feature_count": len(rank55_features),
        "split": split_name,
        "train_n": int(len(y_train)),
        "test_n": int(len(y_test)),
        "accuracy": float(acc),
        "accuracy_percent": float(100 * acc),
        "errors": errors,
        "elapsed_sec": float(elapsed),
    })
    confusion_matrices[split_name] = cm
    models[split_name] = model

results = pd.DataFrame(result_rows)
results


,feature_set,feature_count,split,train_n,test_n,accuracy,accuracy_percent,errors,elapsed_sec
0,top4_rank55,55,2018train20,79720,143912,0.999090,99.908972,131,3.477590
1,top4_rank55,55,2022train20,80000,143632,0.999018,99.901832,141,2.247618


In [7]:
for _, row in results.iterrows():
    split_name = row["split"]
    print("=" * 100)
    print(
        f"{split_name}: accuracy={row['accuracy_percent']:.6f}% | "
        f"errors={int(row['errors'])} / {int(row['test_n'])} | "
        f"train={int(row['train_n'])}"
    )
    print(confusion_matrices[split_name].to_string())
    print()

assert (results["accuracy"] >= 0.998).all(), results


2018train20: accuracy=99.908972% | errors=131 / 143912 | train=79720
            BPSK    MSK  DQPSK   8PSK  QPSK-like
BPSK       17989      0      0      0          0
MSK            0  17989      0      0          0
DQPSK          0      0  17987      0          2
8PSK           0      1      0  17918         70
QPSK-like      0      0     46     12      71898

2022train20: accuracy=99.901832% | errors=141 / 143632 | train=80000
            BPSK    MSK  DQPSK   8PSK  QPSK-like
BPSK       17954      0      0      0          0
MSK            0  17954      0      0          0
DQPSK          0      0  17953      1          0
8PSK           0      0      1  17949          4
QPSK-like      0      0     25    110      71681

